# QTrans—ST-AWFD D2 冻结嵌套正式评估

本 Notebook 只读取已生成的 `frozen_nested_selection.json`。它不运行候选搜索，不根据外层测试结果改变模型配置，并使用全新训练种子 `142, 152, 162, 172, 182`。

## 统计单位和报告边界

每个种子的五个外折预测拼接为一次覆盖474个平衡晶圆的 OOF 结果。论文统计单位是5个训练种子，不把五个外折当成额外独立重复。该任务是从 ST-AWFD D2 派生的监督式评估，不与原论文的无监督异常检测结果直接比较。

In [ ]:
from pathlib import Path
PROJECT_DIR = Path.cwd().resolve()
required_modules = [
    PROJECT_DIR / 'qcs_st_awfd_d2.py',
    PROJECT_DIR / 'qcs_balanced_binary.py',
    PROJECT_DIR / 'qcs_balanced_nested_tuning.py',
    PROJECT_DIR / 'qcs_balanced_nested_formal.py',
]
missing_modules = [str(path) for path in required_modules if not path.exists()]
if missing_modules:
    raise FileNotFoundError('服务器缺少依赖文件：\n' + '\n'.join(missing_modules))

import pandas as pd
import matplotlib.pyplot as plt
import torch
import qcs_balanced_nested_formal as formal_module

from qcs_balanced_binary import paired_comparisons
from qcs_balanced_nested_formal import (
    NESTED_FORMAL_SEEDS, audit_nested_formal, formal_progress,
    require_formal_complete, run_st_awfd_d2_nested_formal,
)

EXPECTED_FORMAL_VERSION = '2026-08-29-st-awfd-d2-v2'
actual_version = getattr(formal_module, 'NESTED_FORMAL_CODE_VERSION', None)
if actual_version != EXPECTED_FORMAL_VERSION:
    raise RuntimeError(
        '服务器 qcs_balanced_nested_formal.py 不是 ST-AWFD D2 新版。'
        '请同步三个 Python 模块，重启 Kernel 后从头运行。'
    )
pd.set_option('display.max_columns', 100)
if not (PROJECT_DIR / 'qcs_balanced_nested_formal.py').exists():
    raise RuntimeError('请从 /root/xxx/autodl 目录打开本 Notebook')
DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'ST-AWFD_D2'
TUNING_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_nested_tuning'
FROZEN = TUNING_ROOT / 'st_awfd_d2' / 'frozen_nested_selection.json'
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_nested_formal'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not FROZEN.exists():
    raise FileNotFoundError(FROZEN)
display(pd.DataFrame([{'device': str(DEVICE), 'formal_seeds': NESTED_FORMAL_SEEDS, 'frozen': str(FROZEN)}]))

## 1. 冻结配置、数据群体和参数量审计

审计会核对冻结协议、候选ID、固定 epoch、参数量差小于1%、数据哈希，以及监督群体是否仍为出版方 `is_test=1` 的604个晶圆。

In [ ]:
audit = audit_nested_formal(
    'st_awfd_d2', DATA_DIR, FROZEN, balance_seed=2026, outer_seed=4096
)
display(pd.DataFrame([audit['dataset']]))
display(audit['selections'])
display(audit['parameters'])
print('Frozen SHA-256:', audit['frozen_sha256'])

## 2. 五外折 × 四模型 × 五种子正式训练

共100个任务。每个任务使用对应外折冻结的配置和固定 epoch，在完整外层开发部分训练，只评价一次外层测试折。支持断点续跑。首次可将 `MAX_JOBS=1` 做环境检查，之后恢复 `None`。

In [ ]:
before = formal_progress(ARTIFACT_ROOT, 'st_awfd_d2')
display(before.groupby('model')['complete'].agg(['sum', 'count']))
print(f"已完成 {int(before.complete.sum())}/{len(before)}")

In [ ]:
MAX_JOBS = None
oof_results, fold_results = run_st_awfd_d2_nested_formal(
    data_dir=DATA_DIR, frozen_selection_path=FROZEN,
    artifact_dir=ARTIFACT_ROOT, seeds=NESTED_FORMAL_SEEDS,
    balance_seed=2026, outer_seed=4096,
    max_jobs=MAX_JOBS, device=DEVICE,
)
print(f'折级结果 {len(fold_results)}/100')
print(f'种子级 OOF 结果 {len(oof_results)}/20')
display(oof_results)

## 3. 完整性检查、汇总和配对检验

In [ ]:
after = formal_progress(ARTIFACT_ROOT, 'st_awfd_d2')
require_formal_complete(after)
RESULT_DIR = ARTIFACT_ROOT / 'st_awfd_d2'
oof_results = pd.read_csv(RESULT_DIR / 'oof_results.csv')
summary = pd.read_csv(RESULT_DIR / 'summary.csv')
display(summary)
paired_acc = paired_comparisons(oof_results, metric='accuracy')
paired_f1 = paired_comparisons(oof_results, metric='macro_f1')
paired_acc.to_csv(RESULT_DIR / 'paired_accuracy.csv', index=False)
paired_f1.to_csv(RESULT_DIR / 'paired_macro_f1.csv', index=False)
display(paired_acc)
display(paired_f1)

In [ ]:
ax = summary.set_index('model')[['accuracy_mean', 'macro_f1_mean']].plot.bar(
    figsize=(9, 4), ylim=(0, 1), rot=15, grid=True
)
ax.set_ylabel('score')
ax.set_title('Frozen nested ST-AWFD D2 supervised OOF evaluation')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'formal_accuracy_macro_f1.png', dpi=300, bbox_inches='tight')
plt.show()

## 输出位置

```text
artifacts/balanced_binary_nested_formal/st_awfd_d2/
├── manifest.json
├── formal_indices.npz
├── fold_results.csv
├── oof_results.csv
├── summary.csv
├── paired_accuracy.csv
├── paired_macro_f1.csv
└── formal_accuracy_macro_f1.png
```

不论量子模型是否第一，都必须保留并报告完整结果。